In [1]:
import os
import csv
from pathlib import Path

In [2]:
# ==========================================
# CONFIGURZIONE
# ==========================================
INPUT_DIR = Path("/home/utente/Scrivania/RLINK/rlink_tractography/pre_post")     # cartella con *-M00.csv e *-M03.csv
OUTPUT_DIR = Path("/home/utente/Scrivania/RLINK/rlink_tractography/delta/subjects")   # cartella di output
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Colonne E..M (Python index: 4..12)
COL_START = 4
COL_END = 12
# ==========================================


def detect_format(value: str):
    """
    Dato un valore numerico come stringa, determina il formato
    da usare per il percent change risultante.
    """
    v = value.strip()

    if v == "":
        return None

    # Notazione scientifica
    if "e" in v.lower():
        # es: 1.234000e-05
        mantissa = v.lower().split("e")[0]
        if "." in mantissa:
            decimals = len(mantissa.split(".")[1])
        else:
            decimals = 0
        return f"%.{decimals}e"

    # Decimale normale
    if "." in v:
        decimals = len(v.split(".")[1])
        return f"%.{decimals}f"

    # Intero
    if v.isdigit() or (v.startswith("-") and v[1:].isdigit()):
        return "%d"

    return None


def safe_float(x):
    """Converte in float, se fallisce restituisce None."""
    try:
        return float(x)
    except:
        return None


def process_pair(file_m00, file_m03, output_file):
    """
    Produce un CSV di percent-change basato su due file:
    - file_m00: baseline
    - file_m03: follow-up
    """
    with open(file_m00, newline="") as f:
        reader = list(csv.reader(f, delimiter=";"))
        header_m00 = reader[0]
        rows_m00 = reader[1:]

    with open(file_m03, newline="") as f:
        reader = list(csv.reader(f, delimiter=";"))
        header_m03 = reader[0]
        rows_m03 = reader[1:]

    # Verifica stesso numero di righe
    if len(rows_m00) != len(rows_m03):
        raise ValueError(f"I file {file_m00} e {file_m03} hanno un numero diverso di righe.")

    # Preparazione output
    output_rows = [header_m03]   # usa header del M03

    for r in range(len(rows_m00)):
        row00 = rows_m00[r]
        row03 = rows_m03[r]
        out_row = row03.copy()

        for c in range(len(row03)):

            # Processiamo solo E..M
            if COL_START <= c <= COL_END:
                v00_str = row00[c].strip()
                v03_str = row03[c].strip()

                v00_num = safe_float(v00_str)
                v03_num = safe_float(v03_str)

                # Caso: M00 non numerico → copiamo M03
                if v00_num is None:
                    out_row[c] = v03_str
                    continue

                # Caso: M00 == 0 → copia esatta del valore M00
                if v00_num == 0:
                    out_row[c] = v00_str
                    continue

                # Caso normale: calcola percent change
                val = (v03_num - v00_num) / v00_num

                fmt = detect_format(v00_str)
                if fmt:
                    out_row[c] = fmt % val
                else:
                    out_row[c] = "%.6g" % val

        output_rows.append(out_row)

    # Scrivi output
    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerows(output_rows)


# ==========================================
# LOOP SUI SOGGETTI
# ==========================================
for file_m00 in INPUT_DIR.glob("*-M00.csv"):
    prefix = file_m00.name.replace("-M00.csv", "")
    file_m03 = INPUT_DIR / f"{prefix}-M03.csv"

    if not file_m03.exists():
        print(f"[SKIP] Manca {file_m03.name}")
        continue

    output_file = OUTPUT_DIR / f"{prefix}-delta.csv"
    print(f"Processing {prefix}...")
    process_pair(file_m00, file_m03, output_file)

print("Fatto!")

Processing Tractometry_64557...
Processing Tractometry_32220...
Processing Tractometry_17468...
Processing Tractometry_50931...
Processing Tractometry_79938...
Processing Tractometry_89532...
Processing Tractometry_71090...
Processing Tractometry_72600...
Processing Tractometry_60036...
Processing Tractometry_29658...
Processing Tractometry_50268...
Processing Tractometry_79627...
Processing Tractometry_52346...
Processing Tractometry_96357...
Processing Tractometry_90396...
Processing Tractometry_43843...
Processing Tractometry_82199...
Processing Tractometry_44245...
Processing Tractometry_38348...
Processing Tractometry_16572...
Processing Tractometry_64091...
Processing Tractometry_57153...
Processing Tractometry_27002...
Processing Tractometry_67834...
Processing Tractometry_59577...
Processing Tractometry_69340...
Processing Tractometry_73150...
Processing Tractometry_27813...
Processing Tractometry_64710...
Processing Tractometry_46618...
Processing Tractometry_87487...
Processi